# News Data Ingestion

This notebook fetches and stores the shared raw stock-news dataset used by all stock notebooks in this project.

It writes one canonical CSV at `news_data/data/news_headlines_raw.csv` and is designed to be run independently from the modeling notebooks.

In [1]:
%pip install -q pandas numpy requests yfinance kagglehub

Note: you may need to restart the kernel to use updated packages.


In [13]:
import os
import json
import time
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests
import yfinance as yf
import kagglehub

pd.set_option('display.max_colwidth', 120)
np.random.seed(42)

def resolve_news_root():
    cwd = Path.cwd()
    if cwd.name == 'implementation' and cwd.parent.name == 'news_data':
        return cwd.parent
    if cwd.name == 'news_data':
        return cwd
    if (cwd / 'project_folder' / 'news_data').exists():
        return cwd / 'project_folder' / 'news_data'
    if (cwd.parent / 'project_folder' / 'news_data').exists():
        return cwd.parent / 'project_folder' / 'news_data'
    if (cwd / 'news_data').exists():
        return cwd / 'news_data'
    if (cwd.parent / 'news_data').exists():
        return cwd.parent / 'news_data'
    return cwd / 'news_data'

news_root = resolve_news_root()
workspace_root = news_root.parent
data_dir = news_root / 'data'
data_dir.mkdir(parents=True, exist_ok=True)
raw_news_path = data_dir / 'news_headlines_raw.csv'

state_path = data_dir / 'news_ingestion_state.json'
log_path = data_dir / 'news_ingestion.log'
stop_flag_path = data_dir / 'stop.flag'

cluster_data_dir = workspace_root / '03_stock_clustering_analysis' / 'data'
reg_data_dir = workspace_root / '02_stock_price_regression' / 'data'
price_candidates = [
    cluster_data_dir / 'sp500_raw.csv',
    reg_data_dir / 'sp500_raw.csv',
    reg_data_dir / '02C_sp500_raw.csv',
    reg_data_dir / '02B_sp500_raw.csv',
]

price_path = next((p for p in price_candidates if p.exists()), None)
if price_path is None:
    print('[INFO] Shared price cache not found locally; downloading reference S&P 500 dataset.')
    kaggle_path = kagglehub.dataset_download('camnugent/sandp500')
    csv_file = [f for f in os.listdir(kaggle_path) if f.endswith('.csv')][0]
    price_df = pd.read_csv(os.path.join(kaggle_path, csv_file))
else:
    price_df = pd.read_csv(price_path)

if 'Name' not in price_df.columns or 'date' not in price_df.columns:
    raise ValueError('Price data must contain Name and date columns.')

price_df = price_df.copy()
price_df['date'] = pd.to_datetime(price_df['date'], errors='coerce').dt.normalize()
price_df = price_df.dropna(subset=['Name', 'date'])
price_df['Name'] = price_df['Name'].astype(str).str.strip()
price_df = price_df.sort_values(['Name', 'date']).reset_index(drop=True)

symbols = sorted(price_df['Name'].dropna().unique().tolist())
min_date = price_df['date'].min()
max_date = price_df['date'].max()

WINDOW_MODE = 'price_aligned'  # 'price_aligned' or 'recent'
NEWS_LOOKBACK_DAYS = 365       # used only when WINDOW_MODE='recent'
NEWS_BUFFER_DAYS = 30          # set 10 or 30: shared news starts at min_date - buffer

if WINDOW_MODE == 'price_aligned':
    window_start = (min_date - pd.Timedelta(days=NEWS_BUFFER_DAYS)).normalize()
    window_end = max_date.normalize()
else:
    today_utc = pd.Timestamp.now(tz='UTC').normalize().tz_localize(None)
    window_end = today_utc
    window_start = (window_end - pd.Timedelta(days=NEWS_LOOKBACK_DAYS)).normalize()

# News fetch controls
NEWS_STOCK_LIMIT = None  # set None to use all selected stocks (slow)
MAX_NEWS_PER_STOCK = None
SLEEP_SEC = 0.05
USE_NEWSAPI = True
USE_GOOGLE_RSS = True
USE_YFINANCE = True
USE_GDELT = False        # GDELT is slow for large windows
NEWSAPI_KEY = os.getenv('NEWSAPI_KEY', '').strip()
use_newsapi_runtime = USE_NEWSAPI and bool(NEWSAPI_KEY)

SAVE_EVERY = 1
LOG_EVERY = 1

if NEWS_STOCK_LIMIT is not None:
    symbols = symbols[:NEWS_STOCK_LIMIT]

def log(msg):
    ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    line = f'[{ts}] {msg}'
    print(line)
    with open(log_path, 'a', encoding='utf-8') as f:
        f.write(line + '\n')

def normalize_text(value):
    return ' '.join(str(value).split()).strip()

def parse_date(value):
    if value is None or value == '':
        return pd.NaT
    parsed = pd.to_datetime(value, utc=True, errors='coerce')
    if pd.isna(parsed):
        return pd.NaT
    return pd.to_datetime(parsed.date())

def safe_ts_to_date(ts):
    if ts is None or pd.isna(ts):
        return pd.NaT
    try:
        return pd.to_datetime(datetime.fromtimestamp(int(ts), tz=timezone.utc).date())
    except Exception:
        return parse_date(ts)

def chunk_dates(start_dt, end_dt, days=30):
    current = pd.Timestamp(start_dt).normalize()
    end_dt = pd.Timestamp(end_dt).normalize()
    while current <= end_dt:
        chunk_end = min(current + pd.Timedelta(days=days - 1), end_dt)
        yield current, chunk_end
        current = chunk_end + pd.Timedelta(days=1)

def fetch_gdelt_news(symbol, start_dt, end_dt, max_records= None):
    rows = []
    query_terms = [f'{symbol} stock']
    for chunk_start, chunk_end in chunk_dates(start_dt, end_dt, days=14):
        for query in query_terms:
            try:
                _params_raw = {
                    'query': query,
                    'mode': 'ArtList',
                    'format': 'json',
                    'sort': 'DateDesc',
                    'maxrecords': max_records,
                    'startdatetime': chunk_start.strftime('%Y%m%d') + '000000',
                    'enddatetime': chunk_end.strftime('%Y%m%d') + '235959',
                }
                # Exclude None values explicitly: the requests library's behaviour with None
                # params is version-dependent and could send the literal string 'None'.
                params = {k: v for k, v in _params_raw.items() if v is not None}
                response = requests.get('https://api.gdeltproject.org/api/v2/doc/doc', params=params, timeout=15)
                if response.status_code == 429:
                    log('[WARN] GDELT rate limit hit (429). Skipping chunk.')
                    articles = []
                else:
                    payload = response.json() if response.ok else {}
                    articles = payload.get('articles', [])
            except requests.exceptions.SSLError as e:
                log(f'[WARN] SSL error fetching GDELT for {symbol}: {e}')
                articles = []
            except requests.exceptions.RequestException as e:
                log(f'[WARN] Request error fetching GDELT for {symbol}: {e}')
                articles = []

            for article in articles:
                title = normalize_text(article.get('title', ''))
                url = normalize_text(article.get('url', ''))
                seendate = parse_date(article.get('seendate', None))
                if title and pd.notna(seendate):
                    rows.append({
                        'Name': symbol,
                        'date': seendate,
                        'headline': title,
                        'source': 'gdelt',
                        'url': url,
                        'published_at': article.get('seendate', None),
                        'query_symbol': symbol,
                        'query_text': query,
                    })
            time.sleep(0.02)
    return rows

def fetch_yfinance_news(symbol, max_items= None):
    rows = []
    try:
        items = yf.Ticker(symbol.replace('.', '-')).news or []
    except Exception as e:
        log(f'[WARN] yfinance news error for {symbol}: {e}')
        items = []

    for item in items[:max_items]:
        title = normalize_text(item.get('title', ''))
        url = normalize_text(item.get('link', ''))
        seendate = safe_ts_to_date(item.get('providerPublishTime', None))
        if title and pd.notna(seendate):
            rows.append({
                'Name': symbol,
                'date': seendate,
                'headline': title,
                'source': 'yfinance',
                'url': url,
                'published_at': item.get('providerPublishTime', None),
                'query_symbol': symbol,
                'query_text': f'{symbol} stock',
            })
    return rows

def fetch_google_rss_news(symbol, max_items= None):
    rows = []
    try:
        q = requests.utils.quote(f'{symbol} stock')
        url = f'https://news.google.com/rss/search?q={q}&hl=en-US&gl=US&ceid=US:en'
        response = requests.get(url, timeout=15, headers={'User-Agent': 'ML-Finance-Research/1.0 (academic)'})
        if not response.ok:
            return rows
        if len(response.content) > 5_000_000:  # 5 MB cap — guard against XML bomb
            log(f'[WARN] Google RSS response too large ({len(response.content):,} bytes) for {symbol}; skipping')
            return rows

        import xml.etree.ElementTree as ET
        root = ET.fromstring(response.content)
        items = root.findall('./channel/item')[:max_items]

        for item in items:
            title_node = item.find('title')
            link_node = item.find('link')
            pub_node = item.find('pubDate')
            title = normalize_text(title_node.text if title_node is not None and title_node.text else '')
            url = normalize_text(link_node.text if link_node is not None and link_node.text else '')
            pub_text = pub_node.text.strip() if pub_node is not None and pub_node.text else ''
            dts = pd.to_datetime(pub_text, utc=True, errors='coerce') if pub_text else pd.NaT
            seendate = pd.to_datetime(dts.date()) if pd.notna(dts) else pd.NaT
            if title and pd.notna(seendate):
                rows.append({
                    'Name': symbol,
                    'date': seendate,
                    'headline': title,
                    'source': 'google_rss',
                    'url': url,
                    'published_at': pub_text,
                    'query_symbol': symbol,
                    'query_text': f'{symbol} stock',
                })
    except requests.exceptions.SSLError as e:
        log(f'[WARN] SSL error fetching Google RSS for {symbol}: {e}')
        return rows
    except requests.exceptions.RequestException as e:
        log(f'[WARN] Request error fetching Google RSS for {symbol}: {e}')
        return rows
    except Exception as e:
        log(f'[WARN] Unexpected error fetching Google RSS for {symbol}: {e}')
        return rows
    return rows

def fetch_newsapi_news(symbol, api_key, page_size=30):
    if not api_key:
        return []

    rows = []
    try:
        params = {
            'q': f'{symbol} stock',
            'language': 'en',
            'sortBy': 'publishedAt',
            'pageSize': page_size,
            'apiKey': api_key
        }
        response = requests.get('https://newsapi.org/v2/everything', params=params, timeout=15)
        if response.status_code == 401:
            log('[ERROR] NewsAPI returned 401 Unauthorized — check NEWSAPI_KEY')
        elif response.status_code == 429:
            log('[WARN] NewsAPI rate limit hit (429). Consider reducing request frequency.')
        payload = response.json() if response.ok else {}
        articles = payload.get('articles', [])
    except requests.exceptions.SSLError as e:
        log(f'[WARN] SSL error calling NewsAPI: {e}')
        articles = []
    except requests.exceptions.RequestException as e:
        log(f'[WARN] Request error calling NewsAPI: {e}')
        articles = []

    for article in articles:
        title = normalize_text(article.get('title', ''))
        url = normalize_text(article.get('url', ''))
        published_at = article.get('publishedAt', None)
        seendate = parse_date(published_at)
        if title and pd.notna(seendate):
            rows.append({
                'Name': symbol,
                'date': seendate,
                'headline': title,
                'source': 'newsapi',
                'url': url,
                'published_at': published_at,
                'query_symbol': symbol,
                'query_text': f'{symbol} stock',
            })
    return rows

def build_price_proxy_news(price_data, target_symbols):
    rows = []
    for sym in target_symbols:
        df_sym = price_data[price_data['Name'] == sym].sort_values('date').copy()
        if df_sym.empty:
            continue
        if 'close' in df_sym.columns:
            df_sym['ret'] = pd.to_numeric(df_sym['close'], errors='coerce').pct_change().fillna(0.0)
        else:
            df_sym['ret'] = 0.0

        for _, rr in df_sym.iterrows():
            d = pd.to_datetime(rr['date'], errors='coerce')
            if pd.isna(d):
                continue
            r = float(rr['ret']) if pd.notna(rr['ret']) else 0.0
            if r >= 0.02:
                text = f'{sym} rallies strongly as market sentiment improves'
            elif r <= -0.02:
                text = f'{sym} drops sharply amid risk-off sentiment'
            elif r > 0:
                text = f'{sym} edges higher in cautious trading session'
            elif r < 0:
                text = f'{sym} slips slightly with mixed market mood'
            else:
                text = f'{sym} trades flat as investors await catalysts'

            rows.append({
                'Name': sym,
                'date': d.normalize(),
                'headline': text,
                'source': 'price_proxy_fallback',
                'url': '',
                'published_at': '',
                'query_symbol': sym,
                'query_text': f'{sym} stock',
            })
    return rows

def load_existing_raw(path):
    if not path.exists():
        return pd.DataFrame()
    existing = pd.read_csv(path)
    if 'date' in existing.columns:
        existing['date'] = pd.to_datetime(existing['date'], errors='coerce').dt.normalize()
    return existing

def save_state(completed_symbols, started_at, current_symbol=''):
    payload = {
        'started_at': started_at,
        'last_update': datetime.now().isoformat(),
        'window_start': str(window_start.date()),
        'window_end': str(window_end.date()),
        'total_symbols': len(symbols),
        'completed_symbols': sorted(list(completed_symbols)),
        'current_symbol': current_symbol,
        'log_path': str(log_path),
    }
    with open(state_path, 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2)

def load_state():
    if not state_path.exists():
        return set(), None
    try:
        with open(state_path, 'r', encoding='utf-8') as f:
            payload = json.load(f)
        completed = set(payload.get('completed_symbols', []))
        started = payload.get('started_at', None)
        return completed, started
    except Exception:
        return set(), None

def normalize_output(df):
    if len(df) == 0:
        return df
    df = df.copy()
    df['Name'] = df['Name'].astype(str).str.strip()
    df['date'] = pd.to_datetime(df['date'], errors='coerce').dt.normalize()
    df['headline'] = df['headline'].astype(str).map(normalize_text)
    if 'source' not in df.columns:
        df['source'] = 'unknown'
    if 'url' not in df.columns:
        df['url'] = ''
    if 'published_at' not in df.columns:
        df['published_at'] = ''
    if 'query_symbol' not in df.columns:
        df['query_symbol'] = df['Name']
    if 'query_text' not in df.columns:
        df['query_text'] = df['Name']

    df = df.dropna(subset=['Name', 'date', 'headline'])
    df = df[(df['Name'].isin(symbols)) & (df['date'] >= window_start) & (df['date'] <= window_end)].copy()
    df = df.drop_duplicates(subset=['Name', 'date', 'headline', 'source', 'url']).reset_index(drop=True)
    df['ingested_at'] = pd.Timestamp.now(tz='UTC').normalize().tz_localize(None)
    df = df.sort_values(['Name', 'date', 'source', 'headline']).reset_index(drop=True)
    return df

def persist_news(existing_df, new_rows):
    new_df = pd.DataFrame(new_rows) if len(new_rows) > 0 else pd.DataFrame()
    merged = pd.concat([existing_df, new_df], ignore_index=True) if len(existing_df) > 0 else new_df
    merged = normalize_output(merged)
    merged.to_csv(raw_news_path, index=False)
    return merged

def fetch_symbol_news(symbol, start_dt, end_dt):
    rows = []
    counts = {'gdelt': 0, 'yfinance': 0, 'google_rss': 0, 'newsapi': 0}

    if USE_GDELT:
        g = fetch_gdelt_news(symbol, start_dt, end_dt, max_records=MAX_NEWS_PER_STOCK)
        rows.extend(g)
        counts['gdelt'] = len(g)
    if USE_YFINANCE:
        y = fetch_yfinance_news(symbol, max_items=MAX_NEWS_PER_STOCK)
        rows.extend(y)
        counts['yfinance'] = len(y)
    if USE_GOOGLE_RSS:
        r = fetch_google_rss_news(symbol, max_items=MAX_NEWS_PER_STOCK)
        rows.extend(r)
        counts['google_rss'] = len(r)
    if use_newsapi_runtime:
        n = fetch_newsapi_news(symbol, NEWSAPI_KEY, page_size=min(30, MAX_NEWS_PER_STOCK))
        rows.extend(n)
        counts['newsapi'] = len(n)

    return rows, counts

log('=== News ingestion started ===')
log(f'Universe size: {len(symbols)} symbols')
log(f'Price coverage: {min_date.date()} -> {max_date.date()}')
log(f'News coverage target: {window_start.date()} -> {window_end.date()}')
log(f'WINDOW_MODE={WINDOW_MODE}, NEWS_BUFFER_DAYS={NEWS_BUFFER_DAYS}')
log(f'USE_NEWSAPI={USE_NEWSAPI}, API key provided={bool(NEWSAPI_KEY)}, effective NewsAPI fetch={use_newsapi_runtime}')
log(f'USE_GOOGLE_RSS={USE_GOOGLE_RSS}, USE_YFINANCE={USE_YFINANCE}, USE_GDELT={USE_GDELT}')
log('Shared-only mode: no local cache fallback, no synthetic proxy fallback')
log(f'State file: {state_path}')
log(f'Log file: {log_path}')
log(f'To stop safely, create this file: {stop_flag_path}')

existing_news = load_existing_raw(raw_news_path)
if len(existing_news) > 0:
    existing_news = existing_news.copy()
    if 'Name' in existing_news.columns:
        existing_news['Name'] = existing_news['Name'].astype(str).str.strip()
    existing_news = existing_news[(existing_news['Name'].isin(symbols)) & (existing_news['date'] >= window_start) & (existing_news['date'] <= window_end)].copy()
else:
    existing_news = pd.DataFrame()

completed_symbols, previous_started = load_state()
completed_symbols = {s for s in completed_symbols if s in set(symbols)}
pending_symbols = [s for s in symbols if s not in completed_symbols]
run_started = datetime.now().isoformat() if previous_started is None else previous_started
if len(existing_news) == 0 and len(pending_symbols) == 0 and len(symbols) > 0:
    log('State says all symbols are complete, but current window has no rows. Resetting state and refetching all symbols.')
    completed_symbols = set()
    pending_symbols = list(symbols)
    run_started = datetime.now().isoformat()

log(f'Already completed from previous runs: {len(completed_symbols)}')
log(f'Pending symbols this run: {len(pending_symbols)}')

news_rows = []
loop_start = time.time()

try:
    for idx, symbol in enumerate(pending_symbols, start=1):
        symbol_start = time.time()
        save_state(completed_symbols, run_started, current_symbol=symbol)

        if stop_flag_path.exists():
            log('Stop flag detected. Exiting loop safely.')
            break

        symbol_rows, source_counts = fetch_symbol_news(symbol, window_start, window_end)
        news_rows.extend(symbol_rows)
        completed_symbols.add(symbol)

        should_save = (idx % SAVE_EVERY == 0) or (idx == len(pending_symbols))
        if should_save:
            existing_news = persist_news(existing_news, news_rows)
            news_rows = []

        elapsed = time.time() - loop_start
        done = len(completed_symbols)
        total = len(symbols)
        avg_per_symbol = elapsed / max(1, idx)
        remaining = total - done
        eta_sec = int(avg_per_symbol * max(0, remaining))
        eta_time = datetime.now() + pd.Timedelta(seconds=eta_sec)
        eta_str = eta_time.strftime('%Y-%m-%d %H:%M:%S')
        symbol_sec = time.time() - symbol_start

        if (idx % LOG_EVERY == 0) or should_save:
            log(
                f"Progress {done}/{total} | current={symbol} | symbol_time={symbol_sec:.1f}s | "
                f"got={len(symbol_rows)} (gdelt={source_counts['gdelt']}, yf={source_counts['yfinance']}, rss={source_counts['google_rss']}, newsapi={source_counts['newsapi']}) | "
                f"elapsed={int(elapsed)}s | ETA={eta_str} | total_rows={len(existing_news):,}"
            )

        time.sleep(SLEEP_SEC)

except KeyboardInterrupt:
    log('KeyboardInterrupt captured. Saving progress before exit...')
finally:
    if len(news_rows) > 0:
        existing_news = persist_news(existing_news, news_rows)
        news_rows = []

    save_state(completed_symbols, run_started, current_symbol='')

    if len(existing_news) == 0:
        raise RuntimeError('No news rows were collected. Shared-only mode does not use cache/proxy fallback. Check network/API settings and rerun.')

    log(f'[OK] Saved shared raw news to: {raw_news_path}')
    log(f'Rows: {len(existing_news):,}')
    coverage_min = existing_news['date'].min().date()
    coverage_max = existing_news['date'].max().date()
    log(f'Coverage: {coverage_min} -> {coverage_max}')
    log('Source counts (top 10):')
    source_counts = existing_news['source'].value_counts(dropna=False).head(10)
    print(source_counts)
    display(existing_news.head(10))

    if len(completed_symbols) == len(symbols) and stop_flag_path.exists():
        stop_flag_path.unlink(missing_ok=True)

    log('=== News ingestion finished ===')

[2026-04-02 18:15:34] === News ingestion started ===
[2026-04-02 18:15:34] Universe size: 505 symbols
[2026-04-02 18:15:34] Price coverage: 2013-02-08 -> 2018-02-07
[2026-04-02 18:15:34] News coverage target: 2013-01-09 -> 2018-02-07
[2026-04-02 18:15:34] WINDOW_MODE=price_aligned, NEWS_BUFFER_DAYS=30
[2026-04-02 18:15:34] USE_NEWSAPI=True, API key provided=False, effective NewsAPI fetch=False
[2026-04-02 18:15:34] USE_GOOGLE_RSS=True, USE_YFINANCE=True, USE_GDELT=False
[2026-04-02 18:15:34] Shared-only mode: no local cache fallback, no synthetic proxy fallback
[2026-04-02 18:15:34] State file: \\compdrive\Student5\25012923g\COMProfile\Documents\GitHub\ML-in-Finance-Data-Project\project_folder\news_data\data\news_ingestion_state.json
[2026-04-02 18:15:34] Log file: \\compdrive\Student5\25012923g\COMProfile\Documents\GitHub\ML-in-Finance-Data-Project\project_folder\news_data\data\news_ingestion.log
[2026-04-02 18:15:34] To stop safely, create this file: \\compdrive\Student5\25012923g\CO

RuntimeError: No news rows were collected. Shared-only mode does not use cache/proxy fallback. Check network/API settings and rerun.